# LangGraph v3 adapter: `_drive_graph` demo

`backend/app/streaming/langgraph_v3_adapter.py` turns one LangGraph run into a flat stream of
queue items (`{"event": ..., "data": ...}`). This notebook calls `_drive_graph` directly, with
the same tools-and-model shape as the app, and splits what comes out into its four components:

| event | produced by | meaning |
|---|---|---|
| `thinking` | `consume_messages` (`message.reasoning`) | model reasoning deltas |
| `token` | `consume_messages` (`message.text`) | the visible answer, as deltas |
| `tool_call` | `consume_messages` (`await message.tool_calls`) | the model's *request* to call a tool |
| `tool_result` | `consume_updates` (`stream.updates`, `ToolMessage`) | proof the tool *ran*, matched by id |

Needs `OPENROUTER_API_KEY` (env var or the project `.env`). No Postgres or Docker needed:
the graph here uses an in-memory checkpointer and two small local tools.

In [1]:
import asyncio, os, sys, time
from collections import defaultdict
from pathlib import Path

# make `app.*` importable (the notebook lives in <repo>/notebooks)
BACKEND = Path.cwd().parent / "backend"
sys.path.insert(0, str(BACKEND))

# read OPENROUTER_API_KEY from the environment or the repo's .env
for env_file in (Path.cwd().parent / ".env", BACKEND / ".env"):
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            if "=" in line and not line.lstrip().startswith("#"):
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())

assert os.environ.get("OPENROUTER_API_KEY"), "set OPENROUTER_API_KEY"
MODEL = os.environ.get("OPENROUTER_MODEL", "anthropic/claude-sonnet-4.5")
EFFORT = os.environ.get("REASONING_EFFORT", "high")
MODEL, EFFORT

('anthropic/claude-sonnet-4.5', 'medium')

## 1. Build a small agent graph

Same construction as `app/agent.py` (`ChatOpenRouter` + `create_agent`), but with two toy tools
whose latency differs on purpose: `slow_lookup` sleeps, `add` is instant. If the model calls both
in one turn, their `tool_result`s arrive out of order, which is exactly why results are matched by id.

In [2]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openrouter import ChatOpenRouter
from langgraph.checkpoint.memory import InMemorySaver

from app.utils.reasoning_merge import ReasoningDetailsMergeCallback


@tool
async def add(a: float, b: float) -> str:
    """Add two numbers."""
    return str(a + b)


@tool
async def slow_lookup(topic: str) -> str:
    """Look up a short fact about a topic (slow, takes ~2 seconds)."""
    await asyncio.sleep(2)
    return f"Fact about {topic}: it was discussed in a notebook."


model = ChatOpenRouter(
    model=MODEL,
    openrouter_api_key=os.environ["OPENROUTER_API_KEY"],
    reasoning={"effort": EFFORT},
    streaming=True,
    model_kwargs={"parallel_tool_calls": True},
    max_retries=3,
    callbacks=[ReasoningDetailsMergeCallback()],
)
graph = create_agent(
    model=model,
    tools=[add, slow_lookup],
    system_prompt="Use the tools whenever they apply. Call independent tools in the same turn.",
    checkpointer=InMemorySaver(),
)

## 2. Run `_drive_graph` and capture every queue item

This is the same producer/consumer shape as `routers/chat.py`: `_drive_graph` is the producer
(a background task), and the loop below is the consumer. Each item is timestamped relative to the
start so you can see the interleaving.

In [3]:
from app.streaming.langgraph_v3_adapter import _drive_graph

PROMPT = "What is 1234 + 8765? Also look up a fact about the moon. Use both tools."

async def capture(prompt: str, thread_id: str):
    queue: asyncio.Queue = asyncio.Queue()
    payload = {"messages": [("user", prompt)]}
    config = {"configurable": {"thread_id": thread_id}}
    t0 = time.perf_counter()

    async def producer():
        try:
            await _drive_graph(graph, payload, config, queue)
        finally:
            await queue.put(None)   # same sentinel run_and_translate uses

    task = asyncio.create_task(producer())
    items = []
    while (item := await queue.get()) is not None:
        items.append({"t": round(time.perf_counter() - t0, 2), **item})
    await task                      # re-raise any exception from the producer
    return items

items = await capture(PROMPT, thread_id="demo-1")
len(items), {e: sum(i["event"] == e for i in items) for e in ("thinking", "token", "tool_call", "tool_result")}

/Users/anupam/Library/Mobile Documents/com~apple~CloudDocs/Documents/Documents - Anupam’s MacBook Air/Codebase/Studies/claude/backend/app/streaming/langgraph_v3_adapter.py:80: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  stream = await graph.astream_events(
/Users/anupam/.venvs/claude-nb/lib/python3.12/site-packages/langgraph/pregel/main.py:3677: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return AsyncGraphRunStream(graph_aiter, mux)


(33, {'thinking': 19, 'token': 10, 'tool_call': 2, 'tool_result': 2})

## 3. Raw timeline (arrival order)

Deltas are collapsed into runs so the order is readable: consecutive `thinking` or `token` items
show up as one row.

In [4]:
def timeline(items):
    rows = []
    for it in items:
        ev, d = it["event"], it["data"]
        if rows and rows[-1]["event"] == ev and ev in ("thinking", "token"):
            rows[-1]["text"] += d["delta"]; rows[-1]["n"] += 1
            continue
        if ev in ("thinking", "token"):
            rows.append({"t": it["t"], "event": ev, "n": 1, "text": d["delta"]})
        else:
            rows.append({"t": it["t"], "event": ev, "n": 1, "text": str(d)})
    return rows

for r in timeline(items):
    text = r["text"].replace("\n", " ")
    print(f"{r['t']:>6}s  {r['event']:<11} x{r['n']:<3} {text[:110]}")

  4.13s  thinking    x19  The user is asking me to: 1. Add 1234 + 8765 using the add function 2. Look up a fact about the moon using the
   5.9s  tool_call   x1   {'id': 'toolu_bdrk_01M3NqQPp4GKNVppTDjagYBF', 'name': 'add', 'args': {'a': 1234, 'b': 8765}}
   5.9s  tool_call   x1   {'id': 'toolu_bdrk_01G5b45r5oFiGxAjGvHT83aF', 'name': 'slow_lookup', 'args': {'topic': 'moon'}}
  5.91s  tool_result x1   {'id': 'toolu_bdrk_01M3NqQPp4GKNVppTDjagYBF', 'result': '9999.0'}
  7.91s  tool_result x1   {'id': 'toolu_bdrk_01G5b45r5oFiGxAjGvHT83aF', 'result': 'Fact about moon: it was discussed in a notebook.'}
 10.45s  token       x10  The answer to 1234 + 8765 is **9999**.  And here's a fact about the moon: it was discussed in a notebook.


## 4. The four components, separately

### 4a. `thinking` — reasoning deltas

Source: `message.reasoning`. May be empty if the model/effort setting returns no reasoning.

In [5]:
thinking = "".join(i["data"]["delta"] for i in items if i["event"] == "thinking")
print(f"{sum(i['event'] == 'thinking' for i in items)} deltas\n")
print(thinking or "(no reasoning returned by this model)")

19 deltas

The user is asking me to:
1. Add 1234 + 8765 using the add function
2. Look up a fact about the moon using the slow_lookup function

Both of these tasks are independent of each other, so I can make both function calls in the same turn.

For the add function:
- a = 1234
- b = 8765

For the slow_lookup function:
- topic = "moon"

Let me call both functions now.


### 4b. `tool_call` — what the model *asked* for

Source: `await message.tool_calls`, emitted once the model message is finalized. This is a request, not a result.

In [6]:
tool_calls = [i for i in items if i["event"] == "tool_call"]
for i in tool_calls:
    print(f"{i['t']:>6}s  id={i['data']['id']}  {i['data']['name']}({i['data']['args']})")

   5.9s  id=toolu_bdrk_01M3NqQPp4GKNVppTDjagYBF  add({'a': 1234, 'b': 8765})
   5.9s  id=toolu_bdrk_01G5b45r5oFiGxAjGvHT83aF  slow_lookup({'topic': 'moon'})


### 4c. `tool_result` — what the tools *returned*

Source: `stream.updates` (the opt-in `UpdatesTransformer` projection), scanning node output for `ToolMessage`s. Note the arrival times: results land as each tool finishes, independent of the order the calls were requested in.

In [7]:
tool_results = [i for i in items if i["event"] == "tool_result"]
for i in tool_results:
    print(f"{i['t']:>6}s  id={i['data']['id']}  -> {i['data']['result']!r}")

  5.91s  id=toolu_bdrk_01M3NqQPp4GKNVppTDjagYBF  -> '9999.0'
  7.91s  id=toolu_bdrk_01G5b45r5oFiGxAjGvHT83aF  -> 'Fact about moon: it was discussed in a notebook.'


### 4d. `token` — the visible assistant message

Source: `message.text`. With tool use there are two model messages: the first usually has no text (just tool calls), the second is the final answer.

In [8]:
message = "".join(i["data"]["delta"] for i in items if i["event"] == "token")
print(f"{sum(i['event'] == 'token' for i in items)} deltas\n")
print(message)

10 deltas

The answer to 1234 + 8765 is **9999**.

And here's a fact about the moon: it was discussed in a notebook.


## 5. Stitching call ↔ result by id

The two streams come from independent consumers (`asyncio.gather` in `_drive_graph`), so nothing
guarantees ordering between them. The id is the only join key, which is how the frontend pairs them.

In [9]:
by_id = defaultdict(dict)
for i in items:
    if i["event"] == "tool_call":
        by_id[i["data"]["id"]].update(name=i["data"]["name"], args=i["data"]["args"], called_at=i["t"])
    elif i["event"] == "tool_result":
        by_id[i["data"]["id"]].update(result=i["data"]["result"], returned_at=i["t"])

for call_id, row in by_id.items():
    took = round(row.get("returned_at", 0) - row.get("called_at", 0), 2)
    print(f"{call_id}\n  {row['name']}({row['args']}) -> {row.get('result')!r}   [{took}s between events]\n")

toolu_bdrk_01M3NqQPp4GKNVppTDjagYBF
  add({'a': 1234, 'b': 8765}) -> '9999.0'   [0.01s between events]

toolu_bdrk_01G5b45r5oFiGxAjGvHT83aF
  slow_lookup({'topic': 'moon'}) -> 'Fact about moon: it was discussed in a notebook.'   [2.01s between events]



## 6. Bonus: the full wrapper (`run_and_translate`)

`_drive_graph` only emits the four content events. `run_and_translate` adds the terminal `done`
(or `error`) event and the `None` sentinel that ends the consumer loop in `routers/chat.py`.

In [10]:
from app.streaming.langgraph_v3_adapter import run_and_translate

queue: asyncio.Queue = asyncio.Queue()
task = asyncio.create_task(run_and_translate(
    graph, {"messages": [("user", "What is 2 + 2? Use the add tool.")]},
    {"configurable": {"thread_id": "demo-2"}}, queue,
))
seen = []
while (item := await queue.get()) is not None:
    seen.append(item["event"])
await task
# collapse consecutive duplicates for readability
[e for n, e in enumerate(seen) if n == 0 or e != seen[n - 1]]

['thinking', 'tool_call', 'tool_result', 'token', 'done']